# Generate VHH sequences from seed sequences.

The purpose of this notebook is to generate VHH sequences ('colonies') from 'seed' sequences.

In [1]:
import random
import pathlib
from itertools import combinations
from pathlib import Path

from collections import defaultdict
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import cm
# from sklearn.preprocessing import OneHotEncoder
import umap

from utils.helpers import (
    hamming_distance,
    one_hot_encode,
    compute_multi_condition_pareto,
    plot_round_radar,
)

# from tqdm.notebook import tqdm

/home/corey.taylor/miniconda3/envs/docking_md/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Protein and data variables.

In [2]:
readout = 'vhh_sequences'

In [3]:
# define paths
# data
HERE = Path(pathlib.Path.cwd())
DATA = HERE / f"data_{readout}"
DATA.mkdir(parents=True, exist_ok=True)

# plots
PLOTS = HERE / f"plots_{readout}"
PLOTS.mkdir(parents=True, exist_ok=True)

## Generate VHH 'colonies' from 'seed' sequences.

### Setup.

In [4]:
samples_per_seed = 100  # how many random mutants per 'seed' sequence
max_mutants = 3                 # maximum number of mutants per sequence
top_candidates_per_round = 100          # top candidates selected per round
n_rounds = 5              # number of rounds for simulation

batch_size = 32 # batch size to feed into the ESM model.

# Define all properties to consider for multi-condition optimisation
multi_condition_props = [
    "Tm_no_heparin","Tm_heparin",
    "solubility_no_heparin","solubility_heparin",
    "expression_no_heparin","expression_heparin"
]

In [5]:
#load dataset
df_seeds = pd.read_csv(f"{DATA}/fake_vhh_multi_conditions.csv")
seeds = df_seeds["sequence"].tolist()
seed_ids = list(range(len(seeds)))

In [6]:
#Define regions (CDR1, CDR2, etc.) and mutable positions in sequences based on position
seq_len = len(seeds[0])
framework_positions = list(range(0, 30)) + list(range(45, 60)) + list(range(67, 95)) + list(range(115, 120)) # only mutating CDRs - check these match real biology
mutable_positions = list(set(range(seq_len)) - set(framework_positions)) # subtract non-mutable regions from complete sequence - tells us which regions we can randomly mutate. 
aa_list = list("ACDEFGHIKLMNPQRSTVWY") # alphabet of AAs we can mutate sequences with.
aa_to_idx = {aa:i for i, aa in enumerate(aa_list)}

### Functions

In [7]:
# function to generate mutants
def generate_mutants(seq, max_mut=max_mutants, sample_per_seed=samples_per_seed):
    mutants = set()
    for k in range(1, max_mut+1):
        for positions in combinations(mutable_positions, k): # super important - sampling by region, rather than randomly over the whole sequence.
            new_seq = list(seq)
            for p in positions:
                new_seq[p] = random.choice(aa_list)
            mutants.add(''.join(new_seq))
            if len(mutants) >= sample_per_seed:
                break
        if len(mutants) >= sample_per_seed:
            break
    return list(mutants)

In [8]:
# def fake_predict(seq):
#     # Return dict of predicted properties
#     return {
#         "Tm_no_heparin": random.uniform(55,75),
#         "Tm_heparin": random.uniform(55,75),
#         "solubility_no_heparin": random.uniform(0.5,1.0),
#         "solubility_heparin": random.uniform(0.5,1.0),
#         "expression_no_heparin": random.uniform(0.2,1.0),
#         "expression_heparin": random.uniform(0.2,1.0)
#     }

#### Load pre-trained ML model for each round to be re-trained each round.

In [9]:
import torch
import esm
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

# Load pretrained ESM-1 model
esm_model, alphabet = esm.pretrained.esm1b_t33_650M_UR50S()
batch_converter = alphabet.get_batch_converter()

esm_model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
esm_model = esm_model.to(device)

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm1b_t33_650M_UR50S.pt" to /home/corey.taylor/.cache/torch/hub/checkpoints/esm1b_t33_650M_UR50S.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm1b_t33_650M_UR50S-contact-regression.pt" to /home/corey.taylor/.cache/torch/hub/checkpoints/esm1b_t33_650M_UR50S-contact-regression.pt


In [10]:
#instantiate pytorch and model
@torch.no_grad()
def esm_embed(sequences, batch_size=batch_size):
    """
    sequences: list[str]
    returns: np.array of shape (N, 1280)
    """
    all_embeddings = []

    for i in range(0, len(sequences), batch_size):
        batch_seqs = sequences[i:i+batch_size]
        data = [("seq", seq) for seq in batch_seqs]
        _, _, batch_tokens = batch_converter(data)
        batch_tokens = batch_tokens.to(device)

        results = esm_model(
            batch_tokens,
            repr_layers=[33],
            return_contacts=False
        )

        token_reps = results["representations"][33]

        for j, seq in enumerate(batch_seqs):
            emb = token_reps[j, 1:len(seq)+1].mean(0)
            all_embeddings.append(emb.cpu().numpy())

        # run in batches (VRAM)
        del batch_tokens, results, token_reps
        torch.cuda.empty_cache()

    return np.vstack(all_embeddings)


### Mutation and selection loop.

In [11]:
df_seeds["ancestor_id"] = df_seeds.index

In [12]:
current_seeds = seeds.copy()
current_seed_ids = seed_ids.copy()
current_ancestor_ids = df_seeds["ancestor_id"].tolist() # lineage tracking
centroid_history = defaultdict(list) # ancestor_id -> list of (round, umap1, umap2)
round_stats = []

In [13]:
# load seeds from scratch prior to training
df_labeled = df_seeds[
    ["sequence"] + multi_condition_props
].copy()

In [14]:
# Initialize centroid history and ancestor colour mapping
ancestor_color_map = {}

for round_idx in range(1, n_rounds+1):
    print(f"=== Round {round_idx} ===")

    # Train / retrain ML model
    X_train = esm_embed(df_labeled["sequence"].tolist())
    y_train = df_labeled[multi_condition_props].values

    print(f'Training dataset size: {len(X_train)}')

    rf = MultiOutputRegressor(
        RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=3,
            random_state=round_idx,
            n_jobs=-1
        )
    )
    rf.fit(X_train, y_train)

    # Generate candidates 
    candidate_seqs = []
    candidate_meta = []

    seed_id_to_ancestor = dict(zip(current_seed_ids, current_ancestor_ids))
    for sid, seq in zip(current_seed_ids, current_seeds):
        muts = generate_mutants(seq)
        candidate_seqs.extend(muts)
        candidate_meta.extend([
            {"seed_id": sid, "ancestor_id": seed_id_to_ancestor[sid], "parent_round": round_idx}
            for _ in muts
        ])
    df_candidates = pd.DataFrame(candidate_meta)
    df_candidates["sequence"] = candidate_seqs

    print(f'Round {round_idx} dataset size: {len(df_candidates)}')

    #  Mutation counts 
    seed_dict = dict(zip(current_seed_ids, current_seeds))
    df_candidates["mut_count"] = df_candidates.apply(
        lambda row: hamming_distance(row["sequence"], seed_dict[row["seed_id"]]), axis=1
    )

    # One-hot + UMAP 
    X = np.array([one_hot_encode(s, seq_len, aa_list, aa_to_idx) for s in df_candidates["sequence"]])
    reducer = umap.UMAP(n_components=2, random_state=42, metric='hamming')
    X_umap = reducer.fit_transform(X)
    df_candidates["umap1"] = X_umap[:,0]
    df_candidates["umap2"] = X_umap[:,1]

    # Predict properties 
    X_cand = esm_embed(df_candidates["sequence"].tolist())
    preds = rf.predict(X_cand)
    df_preds = pd.DataFrame(preds, columns=multi_condition_props)
    df_candidates = pd.concat([df_candidates.reset_index(drop=True), df_preds], axis=1)

    # Compute Pareto front 
    df_candidates["pareto_flag"] = compute_multi_condition_pareto(df_candidates, multi_condition_props)

    #  Priority score, select top candidates 
    df_candidates["score"] = (
        (df_candidates["Tm_no_heparin"] + df_candidates["Tm_heparin"]) / 2
        - 0.5 * df_candidates["mut_count"]
    )
    
    df_selected = df_candidates.sort_values("score", ascending=False).head(top_candidates_per_round)

    current_seeds = df_selected["sequence"].tolist()
    current_seed_ids = list(range(len(current_seeds)))
    current_ancestor_ids = df_selected["ancestor_id"].tolist()

    # Simulate measurements
    df_measured = df_selected.copy()
    noise_scale = 1.0
    for prop in multi_condition_props:
        df_measured[prop] += np.random.normal(0, noise_scale, size=len(df_measured))
    df_labeled = pd.concat([df_labeled, df_measured[["sequence"] + multi_condition_props]], ignore_index=True)

    #  Track per-round statistics 
    stats = {
        "round": round_idx,
        "median_Tm_no_heparin": df_candidates["Tm_no_heparin"].median(),
        "median_Tm_heparin": df_candidates["Tm_heparin"].median(),
        "mean_score": df_candidates["score"].mean(),
        "pareto_count": df_candidates["pareto_flag"].sum()
    }
    round_stats.append(stats)

    # Save round outputs
    df_candidates.to_csv(f"{DATA}/round{round_idx}_candidates.csv", index=False)
    df_selected.to_csv(f"{DATA}/round{round_idx}_selected_batch.csv", index=False)

    # Assign consistent colours for surviving ancestors
    surviving_ancestors = sorted(df_candidates["ancestor_id"].unique())
    tab_palettes = [cm.tab20.colors, cm.tab20b.colors, cm.tab20c.colors]
    combined_colours = np.vstack(tab_palettes)

    if round_idx == 1:
        # Create a color map for all ancestors that appear at least once
        cmap = cm.get_cmap("tab20b", len(surviving_ancestors))
        ancestor_colour_map = {anc: combined_colours[i % len(combined_colours)] 
                      for i, anc in enumerate(surviving_ancestors)}

    # Per-round UMAP
    plt.figure(figsize=(9,7))
    sns.scatterplot(
        data=df_candidates,
        x="umap1",
        y="umap2",
        color="lightgray",
        s=25,
        alpha=0.35,
        legend=False
    )

    df_pareto = df_candidates[df_candidates["pareto_flag"]]
    # record centroids
    for ancestor_id, df_a in df_pareto.groupby("ancestor_id"):
        if len(df_a) >= 3:
            centroid_history[ancestor_id].append({
                "round": round_idx,
                "umap1": df_a["umap1"].mean(),
                "umap2": df_a["umap2"].mean()
            })

    sns.scatterplot(
        data=df_pareto,
        x="umap1",
        y="umap2",
        hue="ancestor_id",
        palette=ancestor_colour_map,
        s=90,
        alpha=0.95,
        legend="full"
    )

    # Annotate Pareto centroids
    for ancestor_id, df_a in df_pareto.groupby("ancestor_id"):
        cx = df_a["umap1"].mean()
        cy = df_a["umap2"].mean()
        plt.text(
            cx, cy, str(ancestor_id),
            fontsize=11, fontweight="bold",
            ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.8, edgecolor="none")
        )

    plt.title(f"Round {round_idx} UMAP - Pareto Regions by Ancestor")
    plt.xlabel("UMAP1")
    plt.ylabel("UMAP2")
    plt.legend(title="Ancestor ID", bbox_to_anchor=(1.05,1), loc="upper left")
    plt.tight_layout()
    plt.savefig(f"{PLOTS}/round{round_idx}_umap_pareto_ancestors.png", dpi=150)
    plt.close()

    save_path = f"{PLOTS}/round{round_idx}_radar.png"
    plot_round_radar(df_candidates, round_idx, save_path, multi_condition_props)

=== Round 1 ===
Training dataset size: 30
Round 1 dataset size: 3000


/home/corey.taylor/miniconda3/envs/docking_md/lib/python3.11/site-packages/umap/umap_.py:1887: UserWarning: gradient function is not yet implemented for hamming distance metric; inverse_transform will be unavailable
  warn(
/home/corey.taylor/miniconda3/envs/docking_md/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/tmp/ipykernel_3793102/1776854538.py:102: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap("tab20b", len(surviving_ancestors))


=== Round 2 ===
Training dataset size: 130
Round 2 dataset size: 10000


/home/corey.taylor/miniconda3/envs/docking_md/lib/python3.11/site-packages/umap/umap_.py:1887: UserWarning: gradient function is not yet implemented for hamming distance metric; inverse_transform will be unavailable
  warn(
/home/corey.taylor/miniconda3/envs/docking_md/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


=== Round 3 ===
Training dataset size: 230
Round 3 dataset size: 10000


/home/corey.taylor/miniconda3/envs/docking_md/lib/python3.11/site-packages/umap/umap_.py:1887: UserWarning: gradient function is not yet implemented for hamming distance metric; inverse_transform will be unavailable
  warn(
/home/corey.taylor/miniconda3/envs/docking_md/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


=== Round 4 ===
Training dataset size: 330
Round 4 dataset size: 10000


/home/corey.taylor/miniconda3/envs/docking_md/lib/python3.11/site-packages/umap/umap_.py:1887: UserWarning: gradient function is not yet implemented for hamming distance metric; inverse_transform will be unavailable
  warn(
/home/corey.taylor/miniconda3/envs/docking_md/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


=== Round 5 ===
Training dataset size: 430
Round 5 dataset size: 10000


/home/corey.taylor/miniconda3/envs/docking_md/lib/python3.11/site-packages/umap/umap_.py:1887: UserWarning: gradient function is not yet implemented for hamming distance metric; inverse_transform will be unavailable
  warn(
/home/corey.taylor/miniconda3/envs/docking_md/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


#### Plot trajectories of seed sequences for all rounds.

In [15]:
# --- Trajectory plot with same colour mapping ---
plt.figure(figsize=(8,7))

for ancestor_id, history in centroid_history.items():
    if len(history) < 2:
        continue
    hx = [h["umap1"] for h in history]
    hy = [h["umap2"] for h in history]
    rounds = [h["round"] for h in history]
    plt.plot(hx, hy, marker="o", linewidth=2, alpha=0.9, color=ancestor_colour_map[ancestor_id], label=f"Ancestor {ancestor_id}")
    # mark final position
    plt.scatter(hx[-1], hy[-1], s=140, edgecolor="black", zorder=5, color=ancestor_colour_map[ancestor_id])
    plt.text(hx[-1], hy[-1], f"R{rounds[-1]}", fontsize=9, ha="left", va="bottom")

plt.title("Pareto Centroid Trajectories Across Rounds")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Ancestor ID", bbox_to_anchor=(1.05,1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{PLOTS}/ancestor_centroid_trajectories.png", dpi=150)
plt.close()


In [16]:
# Lineage prioritisation summary

all_rounds = []
for r in range(1, n_rounds + 1):
    df_r = pd.read_csv(f"{DATA}/round{r}_candidates.csv")
    df_r["round"] = r
    all_rounds.append(df_r)

df_all = pd.concat(all_rounds, ignore_index=True)

ancestor_summary = (
    df_all
    .groupby("ancestor_id")
    .agg(
        total_descendants=("sequence", "count"),
        pareto_rate=("pareto_flag", "mean"),
        best_score=("score", "max"),
        mean_score=("score", "mean"),
        max_mutations=("mut_count", "max"),
        survived_rounds=("round", "max"),
    )
    .reset_index()
    .sort_values("best_score", ascending=False)
)

ancestor_summary.to_csv(
    f"{DATA}/ancestor_prioritisation_summary.csv",
    index=False
)

print("Top ancestral scaffolds:")
print(ancestor_summary.head(10))


Top ancestral scaffolds:
    ancestor_id  total_descendants  pareto_rate  best_score  mean_score  \
26           26               6900     0.098116   73.209440   69.854567   
9             9              24300     0.098395   72.778723   67.501312   
24           24               8000     0.095375   72.635877   69.646588   
10           10                800     0.068750   71.911810   68.854118   
22           22                300     0.406667   70.921620   67.966932   
21           21                200     0.335000   70.070025   67.397857   
4             4                200     0.310000   69.506587   67.630760   
3             3                100     0.010000   69.067688   67.299069   
18           18                100     0.000000   68.664664   66.793853   
1             1                100     0.000000   68.359074   66.351389   

    max_mutations  survived_rounds  
26              2                5  
9               2                5  
24              2                5  
1

In [17]:
df_stats = pd.DataFrame(round_stats)

# Median Tm
plt.figure(figsize=(8,5))
plt.plot(df_stats["round"], df_stats["median_Tm_no_heparin"], marker='o', label="Tm no heparin")
plt.plot(df_stats["round"], df_stats["median_Tm_heparin"], marker='o', label="Tm heparin")
plt.xlabel("Round")
plt.ylabel("Median Tm (°C)")
plt.title("Round-over-round property improvement")
plt.legend()
plt.tight_layout()
plt.savefig(f"{PLOTS}/round_over_round_Tm.png", dpi=150)
plt.close()

# Mean score
plt.figure(figsize=(8,5))
plt.plot(df_stats["round"], df_stats["mean_score"], marker='o', color='green', label="Mean score")
plt.xlabel("Round")
plt.ylabel("Mean priority score")
plt.title("Round-over-round priority score evolution")
plt.tight_layout()
plt.savefig(f"{PLOTS}/round_over_round_score.png", dpi=150)
plt.close()

# Pareto front count
plt.figure(figsize=(8,5))
plt.plot(df_stats["round"], df_stats["pareto_count"], marker='o', color='red', label="Pareto front count")
plt.xlabel("Round")
plt.ylabel("Number of Pareto-optimal candidates")
plt.title("Round-over-round Pareto front coverage")
plt.tight_layout()
plt.savefig(f"{PLOTS}/round_over_round_pareto_count.png", dpi=150)
plt.close()